# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [ ]:
# ── 0. Install libraries ──────────────────────────────────────────────────────
!pip install -q transformers==4.49.0 peft==0.14.0 bitsandbytes accelerate datasets pillow

In [ ]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import glob
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths (auto-detects Kaggle vs local) ─────────────────────────────────────
_kaggle_input = Path("/kaggle/input")
if _kaggle_input.exists():
    # Debug: show everything mounted under /kaggle/input
    print("Contents of /kaggle/input:")
    for _entry in sorted(_kaggle_input.rglob("*"))[:40]:   # cap at 40 lines
        print(f"  {_entry}")

    # Search recursively for train.csv
    _matches = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    print(f"\nFound train.csv at: {_matches}")
    if not _matches:
        raise RuntimeError(
            "train.csv not found under /kaggle/input. "
            "Make sure the competition dataset is attached to this notebook."
        )
    # DATA_DIR is the folder that contains train.csv
    DATA_DIR = Path(_matches[0]).parent
    print(f"DATA_DIR set to: {DATA_DIR}")
else:
    DATA_DIR = Path("data")      # local development path

# ── Model — offline-safe ─────────────────────────────────────────────────────
# Attach the SmolVLM Kaggle dataset and name it "smolvlm-500m-instruct" so the
# notebook works without internet access during evaluation.
_HF_MODEL_ID  = "HuggingFaceTB/SmolVLM-500M-Instruct"
_local_model  = Path("/kaggle/input/smolvlm-500m-instruct")
MODEL_ID      = str(_local_model) if _local_model.exists() else _HF_MODEL_ID
print(f"Model source: {MODEL_ID}")

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE = 224

# SmolVLM expands <image> into ~1088 placeholder tokens in input_ids.
# Truncating the combined sequence cuts those placeholders and causes a shape
# mismatch in inputs_merger. Instead we cap the TEXT portion in build_prompt.
MAX_CONTEXT_CHARS = 400   # per field (lecture / hint) — ~100 tokens each

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load and Preprocess Data

In [3]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

Train: 6,218 | Val: 2,097 | Test: 2,017


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_00000,images/train/train_00000.png,Which of these states is farthest north?,"[West Virginia, Louisiana, Arizona, Oklahoma]",4,0,NaN,"Maps have four cardinal directions, or main di...","To find the answer, look at the compass rose. ...",closed choice,grade2,social science,geography,Geography,Read a map: cardinal directions
1,train_00001,images/train/train_00001.png,Identify the question that Tom and Justin's ex...,[Do ping pong balls stop rolling along the gro...,2,1,The passage below describes an experiment. Rea...,Experiments can be designed to answer specific...,NaN,closed choice,grade8,natural science,science-and-engineering-practices,Designing experiments,Identify the experimental question


In [ ]:
# ── 2a-debug. Show what's actually mounted under DATA_DIR ─────────────────────
import os

print(f"DATA_DIR = {DATA_DIR}\n")
print("Top-level contents:")
for entry in sorted(DATA_DIR.iterdir()):
    size = f"  ({sum(1 for _ in entry.rglob('*'))} files)" if entry.is_dir() else ""
    print(f"  {'[DIR]' if entry.is_dir() else '[FILE]'} {entry.name}{size}")

# Search for any PNG to confirm images are accessible
_sample_pngs = list(DATA_DIR.rglob("*.png"))[:5]
print(f"\nSample PNG paths found ({len(list(DATA_DIR.rglob('*.png')))} total):")
for p in _sample_pngs:
    print(f"  {p}")

In [ ]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Builds the text prompt for the Vision Language Model.
    The <image> token is required for the model to process the image.
    Lecture/hint are capped at MAX_CONTEXT_CHARS to keep total sequence length
    bounded — SmolVLM image placeholders must not be truncated by the processor.
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        ctx = str(lecture).strip()
        if len(ctx) > MAX_CONTEXT_CHARS:
            ctx = ctx[:MAX_CONTEXT_CHARS]
        context_parts.append(ctx)
    if pd.notna(hint) and str(hint).strip():
        ctx = str(hint).strip()
        if len(ctx) > MAX_CONTEXT_CHARS:
            ctx = ctx[:MAX_CONTEXT_CHARS]
        context_parts.append(ctx)
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

# Display an example prompt
print(build_prompt(train_df.iloc[0], include_answer=True))

In [ ]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
def _resolve_image_path(data_dir: Path, rel_path: str) -> Path:
    """
    Kaggle mounts competition images under images/images/<split>/ (double-nested),
    but the CSV image_path column uses images/<split>/. Try both plus other fallbacks.
    """
    parts = Path(rel_path).parts   # e.g. ('images', 'val', 'val_00671.png')
    candidates = [
        data_dir / rel_path,                            # images/val/x.png  (local)
        data_dir / "images" / rel_path,                 # images/images/val/x.png  (Kaggle)
        data_dir / Path(rel_path).name,                 # flat: x.png
        data_dir / "images" / Path(rel_path).name,      # images/x.png
    ]
    # strip leading "images/" and try under images/
    if parts[0] == "images" and len(parts) > 1:
        candidates.append(data_dir / Path(*parts[1:]))
        candidates.append(data_dir / "images" / Path(*parts[1:]))

    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "Image not found. Tried:\n" + "\n".join(f"  {p}" for p in candidates)
    )


class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
        self.df       = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        path = _resolve_image_path(self.data_dir, rel_path)
        return Image.open(path).convert("RGB").resize((self.img_size, self.img_size), Image.BICUBIC)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }

train_ds = ScienceQADataset(train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

## 3. Load Processor

Load the processor (tokenizer + image processor) once here. The model itself is loaded later in Section 5 directly in 4-bit NF4 — no need to load fp16 first.

In [ ]:
# ── 3a. Load processor ───────────────────────────────────────────────────────
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print("Processor loaded.")

## 4. Log-Likelihood Inference

Instead of free-form generation (which is unreliable for multiple-choice), we score each answer choice directly:

1. Build the prompt ending with `"Answer:"`
2. Run **one forward pass** through the model
3. At the final token position (which predicts what comes next), extract the logit for each valid choice letter (A, B, C, ...)
4. Return the argmax — i.e., the letter the model assigns highest probability to

This is accurate, fast (one forward pass per question), and handles variable numbers of choices automatically.

In [ ]:
# ── 4a. Inference & evaluation functions ─────────────────────────────────────
import torch.nn.functional as F
from tqdm.auto import tqdm

# Pre-compute token IDs for each choice letter once
_LETTER_TOKEN_IDS = {
    letter: processor.tokenizer.encode(f" {letter}", add_special_tokens=False)[-1]
    for letter in CHOICE_LETTERS
}

def predict_loglikelihood(model, processor, image: Image.Image, prompt_text: str, num_choices: int) -> int:
    """Single forward pass → argmax log-prob over valid choice letters."""
    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt")
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits       # (1, seq_len, vocab_size)

    last_logits = logits[0, -1, :]            # logits at the Answer: position
    log_probs   = F.log_softmax(last_logits, dim=-1)
    scores = [log_probs[_LETTER_TOKEN_IDS[CHOICE_LETTERS[i]]].item() for i in range(num_choices)]
    return int(torch.tensor(scores).argmax().item())


def evaluate(model, processor, df: pd.DataFrame, data_dir: Path, img_size: int = IMG_SIZE):
    """Run log-likelihood inference on a labeled split, return (predictions, accuracy)."""
    model.eval()
    preds, correct = [], 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating"):
        image  = Image.open(_resolve_image_path(data_dir, row["image_path"])).convert("RGB").resize((img_size, img_size))
        prompt = build_prompt(row, include_answer=False)
        pred   = predict_loglikelihood(model, processor, image, prompt, int(row["num_choices"]))
        preds.append(pred)
        correct += int(pred == int(row["answer"]))
    accuracy = correct / len(df)
    print(f"Accuracy: {correct}/{len(df)} = {accuracy:.4f}")
    return preds, accuracy


def generate_submission(model, processor, df: pd.DataFrame, data_dir: Path,
                        img_size: int = IMG_SIZE, output_path: str = "/kaggle/working/submission.csv"):
    """Predict on test set and write submission.csv."""
    model.eval()
    preds = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Predicting test"):
        image  = Image.open(_resolve_image_path(data_dir, row["image_path"])).convert("RGB").resize((img_size, img_size))
        prompt = build_prompt(row, include_answer=False)
        pred   = predict_loglikelihood(model, processor, image, prompt, int(row["num_choices"]))
        preds.append(pred)
    submission = pd.DataFrame({"id": df["id"].values, "answer": preds})
    submission.to_csv(output_path, index=False)
    print(f"Saved {len(submission)} predictions → {output_path}")
    return submission

print("Inference functions defined.")

## 5. QLoRA Fine-Tuning

We reload the model in **4-bit NF4** (via `bitsandbytes`) and attach **LoRA adapters** to the attention projections of the language model only. This keeps trainable parameters well under the 5 M cap while the frozen 4-bit backbone provides full representational capacity.

Key choices:
- `r=8`, `lora_alpha=32` → ~4.6 M trainable params (attention + MLP modules)
- Loss computed only on the **answer letter token** (last token of each training sequence) so the model learns to pick the right choice, not to recite the prompt
- `grad_accum_steps=8`, `lora_dropout=0.1` — Run 9: dropout increased from 0.05 for regularization; grad_accum restored to 8 (accum=4 + alpha=32 crashed Run 8 to 0.79)

In [ ]:
# ── 5a. Load model in 4-bit NF4 + attach LoRA adapters ───────────────────────
from transformers import AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ── Hyperparameters ──────────────────────────────────────────────────────────
LR               = 2e-4
NUM_EPOCHS       = 3
GRAD_ACCUM_STEPS = 8      # Run 9: restored to 8; accum=4+alpha=32 crashed Run 8 (0.79)

# SmolVLM expands <image> into ~1088 placeholder tokens in input_ids.
# Truncating the combined sequence cuts those placeholders and causes a shape
# mismatch in inputs_merger. Instead we cap the TEXT portion in build_prompt.
MAX_CONTEXT_CHARS = 400   # per field (lecture / hint) — ~100 tokens each

LORA_R       = 8    # r=8; attention+MLP modules → ~4.6M trainable params
LORA_ALPHA   = 32   # Run 6: alpha=32 confirmed +0.016 public LB; kept for Run 9
LORA_DROPOUT = 0.1  # Run 9: increased from 0.05 — regularization is the bottleneck

# ── 4-bit quantisation ───────────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# ── LoRA on attention + MLP projections ──────────────────────────────────────
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()   # confirm we're under 5 M

In [ ]:
# ── 5b. Training collate function ────────────────────────────────────────────
# SmolVLM expands <image> into ~1088 image tokens internally.
# Processor-level truncation would cut those placeholders and break inputs_merger.
# Text length is already bounded by MAX_CONTEXT_CHARS in build_prompt, so no
# truncation is needed here.

def collate_train(batch):
    texts  = [item["text"]  for item in batch]   # prompt ending with "Answer: X"
    images = [item["image"] for item in batch]

    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v
              for k, v in inputs.items()}

    # Mask all tokens except the answer letter so loss = -log P(correct letter)
    labels = inputs["input_ids"].clone()
    labels[:, :-1] = -100
    inputs["labels"] = labels
    return inputs

# Combine train + val for maximum training signal on final submission run
combined_df = pd.concat([train_df, val_df], ignore_index=True)
combined_ds = ScienceQADataset(combined_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)

train_loader = DataLoader(
    combined_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=collate_train,
    num_workers=0,
)
print(f"Training on {len(combined_df)} examples ({len(train_df)} train + {len(val_df)} val)")
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
# ── 5c. Training loop (no validation — training on train+val combined) ────────
import time

LOG_EVERY = 50   # print a status line every this many steps

def train_model(model, processor, train_loader,
                num_epochs=NUM_EPOCHS, lr=LR, grad_accum=GRAD_ACCUM_STEPS):

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=0.01,
    )
    use_amp = torch.cuda.is_available()
    scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)

    total_steps = num_epochs * len(train_loader)

    print(f"\n{'='*60}")
    print(f"Training config:")
    print(f"  Epochs:            {num_epochs}")
    print(f"  Steps/epoch:       {len(train_loader)}")
    print(f"  Total steps:       {total_steps}")
    print(f"  Learning rate:     {lr}  (flat)")
    print(f"  Grad accum steps:  {grad_accum}  (effective batch = {grad_accum})")
    print(f"  Max context chars: {MAX_CONTEXT_CHARS}")
    print(f"  AMP enabled:       {use_amp}")
    print(f"{'='*60}\n")

    train_start = time.time()

    for epoch in range(num_epochs):
        model.train()
        total_loss   = 0.0
        epoch_start  = time.time()
        optimizer.zero_grad()

        print(f"--- Epoch {epoch + 1}/{num_epochs} starting ---")

        for step, batch in enumerate(train_loader):

            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                loss = model(**batch).loss / grad_accum

            scaler.scale(loss).backward()
            total_loss += loss.item() * grad_accum

            if (step + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            if (step + 1) % LOG_EVERY == 0 or step == 0:
                avg_loss      = total_loss / (step + 1)
                elapsed       = time.time() - epoch_start
                secs_per_step = elapsed / (step + 1)
                steps_left    = len(train_loader) - (step + 1)
                eta_epoch     = steps_left * secs_per_step
                global_step   = epoch * len(train_loader) + step + 1
                total_elapsed = time.time() - train_start
                total_eta     = (total_steps - global_step) * (total_elapsed / global_step)

                print(
                    f"  [E{epoch+1} step {step+1:>4}/{len(train_loader)}] "
                    f"loss={avg_loss:.4f} | "
                    f"{secs_per_step:.1f}s/step | "
                    f"epoch ETA {eta_epoch/60:.1f}min | "
                    f"total ETA {total_eta/60:.0f}min"
                )

        avg_loss   = total_loss / len(train_loader)
        epoch_time = time.time() - epoch_start
        print(f"\nEpoch {epoch + 1} done in {epoch_time/60:.1f}min | avg train loss: {avg_loss:.4f}")

    total_time = time.time() - train_start
    print(f"\nTraining complete in {total_time/60:.1f}min.")
    return model

In [ ]:
# ── 5d. Run training, save adapter, then generate final submission ────────────
model = train_model(model, processor, train_loader)

# Save trained LoRA adapter + processor to Kaggle output (offline-safe)
ADAPTER_SAVE_PATH = "/kaggle/working/lora_adapter_final"
model.save_pretrained(ADAPTER_SAVE_PATH)
processor.save_pretrained(ADAPTER_SAVE_PATH)
print(f"Adapter saved to {ADAPTER_SAVE_PATH}")

submission = generate_submission(model, processor, test_df, DATA_DIR)
submission.head()